In [1]:
REPO_URL = "https://github.com/MberkKeskin/turkish_legal_rag_assistant.git"
PROJECT_DIR = "/content/turkish_legal_rag_assistant/FINAL_SUBMISSION"

HF_MODELS = {
    "bge-m3-legal-ft-system2": "Berk2003/bge-m3-legal-ft-system2",
    "bge_reranker_legal_ft_v4_error_mined_hf": "Berk2003/bge-reranker-legal-ft-v4-error-mined-hf",
    "qwen2_5_3b_legal_lora_sft_faithful_v2_final": "Berk2003/qwen2-5-3b-legal-lora-sft-faithful-v2-final",
}

print("Repository:", REPO_URL)
print("Project dir:", PROJECT_DIR)
print("Hugging Face models:")
for local_name, repo_id in HF_MODELS.items():
    print(f"  {local_name} <- {repo_id}")

!nvidia-smi || true

Repository: https://github.com/MberkKeskin/turkish_legal_rag_assistant.git
Project dir: /content/turkish_legal_rag_assistant/FINAL_SUBMISSION
Hugging Face models:
  bge-m3-legal-ft-system2 <- Berk2003/bge-m3-legal-ft-system2
  bge_reranker_legal_ft_v4_error_mined_hf <- Berk2003/bge-reranker-legal-ft-v4-error-mined-hf
  qwen2_5_3b_legal_lora_sft_faithful_v2_final <- Berk2003/qwen2-5-3b-legal-lora-sft-faithful-v2-final
Sat Jun 13 11:51:04 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG 

In [2]:
%cd /content
!rm -rf turkish_legal_rag_assistant
!git clone "$REPO_URL"
%cd "$PROJECT_DIR"

!pwd
!ls

/content
Cloning into 'turkish_legal_rag_assistant'...
remote: Enumerating objects: 439, done.
remote: Counting objects: 100% (439/439), done.
remote: Compressing objects: 100% (274/274), done.
remote: Total 439 (delta 182), reused 404 (delta 161), pack-reused 0 (from 0)
Receiving objects: 100% (439/439), 27.28 MiB | 12.15 MiB/s, done.
Resolving deltas: 100% (182/182), done.
Updating files: 100% (398/398), done.
/content/turkish_legal_rag_assistant/FINAL_SUBMISSION
/content/turkish_legal_rag_assistant/FINAL_SUBMISSION
 allTestResults				 FINAL_LEGAL_RAG_SYSTEM
 app					 final_ui.py
 data					 models
 evaluate_base_vs_final_retrieval.py	'README (1).md'
 evaluate_custom_benchmark.py		 README.md
 evaluate_one_physical_stage_worker.py	 requirements.txt
 evaluate_physical_stagewise_20.py	 smoke_test_light.py
 evaluate_real_stagewise_50.py		 smoke_test.py
 evaluate_stagewise_full_20.py		 testResultsFinal
 evaluation_results


In [3]:
!pip install -q -r requirements.txt
!pip install -q -U huggingface_hub pandas ipywidgets

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 71.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 346.6/346.6 kB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 25.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 132.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 693.4/693.4 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 107.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 104.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the pac

In [4]:
from pathlib import Path
from huggingface_hub import snapshot_download
import shutil
import os

os.environ["HF_HUB_DISABLE_XET"] = "1"

models_dir = Path(PROJECT_DIR) / "models"

# GitHub'dan boş/eksik models klasörü geldiyse temizle
if models_dir.exists():
    shutil.rmtree(models_dir)

models_dir.mkdir(parents=True, exist_ok=True)

def count_files(path):
    path = Path(path)
    return len([x for x in path.rglob("*") if x.is_file()]) if path.exists() else 0

def folder_size_mb(path):
    path = Path(path)
    return sum(x.stat().st_size for x in path.rglob("*") if x.is_file()) / (1024 * 1024) if path.exists() else 0

for local_name, repo_id in HF_MODELS.items():
    target_dir = models_dir / local_name

    print("\n" + "=" * 90)
    print("Downloading model")
    print("HF repo   :", repo_id)
    print("Local path:", target_dir)
    print("=" * 90)

    snapshot_download(
        repo_id=repo_id,
        repo_type="model",
        local_dir=str(target_dir),
        local_dir_use_symlinks=False,
    )

    print("Downloaded:", local_name)
    print("Files:", count_files(target_dir))
    print("Size MB:", round(folder_size_mb(target_dir), 2))

print("\nFinal model check:")
for local_name in HF_MODELS:
    p = models_dir / local_name
    print(f"{local_name:60s} exists={p.exists()} files={count_files(p)} size_mb={round(folder_size_mb(p), 2)}")

missing = [
    name for name in HF_MODELS
    if not (models_dir / name).exists() or count_files(models_dir / name) == 0
]

if missing:
    raise RuntimeError(f"Missing model folders: {missing}")

print("\nAll Hugging Face models downloaded successfully.")


HF repo   : Berk2003/bge-m3-legal-ft-system2
Local path: /content/turkish_legal_rag_assistant/FINAL_SUBMISSION/models/bge-m3-legal-ft-system2


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

Downloaded: bge-m3-legal-ft-system2
Files: 22
Size MB: 2182.2

HF repo   : Berk2003/bge-reranker-legal-ft-v4-error-mined-hf
Local path: /content/turkish_legal_rag_assistant/FINAL_SUBMISSION/models/bge_reranker_legal_ft_v4_error_mined_hf


Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

Downloaded: bge_reranker_legal_ft_v4_error_mined_hf
Files: 20
Size MB: 2181.9

HF repo   : Berk2003/qwen2-5-3b-legal-lora-sft-faithful-v2-final
Local path: /content/turkish_legal_rag_assistant/FINAL_SUBMISSION/models/qwen2_5_3b_legal_lora_sft_faithful_v2_final


Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

Downloaded: qwen2_5_3b_legal_lora_sft_faithful_v2_final
Files: 22
Size MB: 68.08

Final model check:
bge-m3-legal-ft-system2                                      exists=True files=22 size_mb=2182.2
bge_reranker_legal_ft_v4_error_mined_hf                      exists=True files=20 size_mb=2181.9
qwen2_5_3b_legal_lora_sft_faithful_v2_final                  exists=True files=22 size_mb=68.08

All Hugging Face models downloaded successfully.


In [5]:
from pathlib import Path

checks = {
    "models/bge-m3-legal-ft-system2": ["model.safetensors"],
    "models/bge_reranker_legal_ft_v4_error_mined_hf": ["model.safetensors"],
    "models/qwen2_5_3b_legal_lora_sft_faithful_v2_final": ["adapter_model.safetensors"],
}

def size_mb(path):
    return path.stat().st_size / (1024 * 1024)

for folder, required_files in checks.items():
    folder_path = Path(folder)
    print("\n" + "=" * 80)
    print(folder)
    print("exists:", folder_path.exists())

    for req in required_files:
        matches = list(folder_path.rglob(req)) if folder_path.exists() else []
        if not matches:
            raise RuntimeError(f"Missing required file: {folder}/{req}")

        for m in matches:
            print(req, "->", m, "| size MB:", round(size_mb(m), 2))

print("\nRequired model files are valid.")


models/bge-m3-legal-ft-system2
exists: True
model.safetensors -> models/bge-m3-legal-ft-system2/model.safetensors | size MB: 2165.86

models/bge_reranker_legal_ft_v4_error_mined_hf
exists: True
model.safetensors -> models/bge_reranker_legal_ft_v4_error_mined_hf/model.safetensors | size MB: 2165.86

models/qwen2_5_3b_legal_lora_sft_faithful_v2_final
exists: True
adapter_model.safetensors -> models/qwen2_5_3b_legal_lora_sft_faithful_v2_final/adapter_model.safetensors | size MB: 57.16

Required model files are valid.


In [6]:
from pathlib import Path

required_items = [
    "README.md",
    "requirements.txt",
    "smoke_test.py",
    "smoke_test_light.py",
    "evaluate_custom_benchmark.py",
    "app",
    "data",
    "models",
]

print("Project file check:")
for item in required_items:
    p = Path(item)
    print(f"{item:40s}", "OK" if p.exists() else "MISSING")

Project file check:
README.md                                OK
requirements.txt                         OK
smoke_test.py                            OK
smoke_test_light.py                      OK
evaluate_custom_benchmark.py             OK
app                                      OK
data                                     OK
models                                   OK


In [7]:
from pathlib import Path

code = r'''
import json
import subprocess
from pathlib import Path

import pandas as pd
from IPython.display import display, clear_output, HTML
import ipywidgets as widgets

BASE_DIR = Path.cwd()

DEMO_BENCHMARK = [
    {
        "question": "Türk Borçlar Kanunu m.314 kapsamında ifa zamanı nasıl düzenlenmiştir?",
        "gold_answer": "Kiracı, aksine sözleşme ve yerel adet olmadıkça kira bedelini ve gerekiyorsa yan giderleri her ayın sonunda ve en geç kira süresinin bitiminde ödemekle yükümlüdür.",
        "gold_ids": ["turkish_law_eski_6098_turk_borclar_kanunu_m314"]
    },
    {
        "question": "Türk Medeni Kanunu'na göre evlilik için yaş şartı nedir?",
        "gold_answer": "Kişi on yedi yaşını doldurmadıkça evlenemez. Ancak olağanüstü durumlarda ve pek önemli bir sebep varsa, on altı yaşını dolduran kişinin evlenmesine hâkim izin verebilir.",
        "gold_ids": ["turkish_law_eski_4721_turk_medeni_kanunu_m124"]
    },
    {
        "question": "Ayırt etme gücüne sahip olmayan bir kişi evlenebilir mi?",
        "gold_answer": "Ayırt etme gücüne sahip olmayan kişiler evlenemez.",
        "gold_ids": ["turkish_law_eski_4721_turk_medeni_kanunu_m125"]
    },
    {
        "question": "Kiracının kira bedelini ödememesi halinde kiraya veren hangi hukuki yola başvurabilir?",
        "gold_answer": "Kiracı kira bedelini ödemezse kiraya veren, kanunda öngörülen şartlara göre kiracıya yazılı bildirimde bulunabilir ve verilen sürede ödeme yapılmazsa sözleşmenin feshi veya tahliye gibi hukuki yollara başvurabilir.",
        "gold_ids": ["turkish_law_eski_6098_turk_borclar_kanunu_m315"]
    },
    {
        "question": "Depozito bakımından konut ve çatılı işyeri kiralarında temel sınırlama nedir?",
        "gold_answer": "Konut ve çatılı işyeri kiralarında depozito miktarı üç aylık kira bedelini aşamaz.",
        "gold_ids": ["turkish_law_eski_6098_turk_borclar_kanunu_m342"]
    }
]


def read_uploaded_json(upload_widget):
    if not upload_widget.value:
        return None, None

    uploaded = upload_widget.value

    if isinstance(uploaded, dict):
        first = next(iter(uploaded.values()))
        content = first["content"]
        filename = first.get("metadata", {}).get("name", "uploaded_benchmark.json")
    else:
        first = uploaded[0]
        content = first["content"]
        filename = first.get("name", "uploaded_benchmark.json")

    text = bytes(content).decode("utf-8")
    data = json.loads(text)

    if isinstance(data, dict):
        data = data.get("data") or data.get("items") or data.get("benchmark")

    if not isinstance(data, list):
        raise ValueError("Benchmark JSON must be a list of question objects.")

    return filename, data


def save_json(data, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)


def run_script(input_path, output_path, mode):
    cmd = [
        "python",
        "evaluate_custom_benchmark.py",
        "--benchmark",
        str(input_path),
        "--output",
        str(output_path),
    ]

    if mode == "retrieval":
        cmd.append("--no_generator")

    return subprocess.run(
        cmd,
        cwd=str(BASE_DIR),
        capture_output=True,
        text=True
    )


def summarize_metrics(df):
    possible_metrics = [
        "Recall@1", "Recall@3", "Recall@5",
        "Recall1", "Recall3", "Recall5",
        "MRR", "Token_F1", "token_f1",
        "EM", "Exact_Match",
        "ROUGE1", "ROUGE_F", "BLEU",
        "Faithfulness", "Hallucination_Risk", "Citation_Accuracy",
        "Final_Rubric_Score", "Final_Rubric_Proxy"
    ]

    rows = []

    for col in possible_metrics:
        if col in df.columns:
            vals = pd.to_numeric(df[col], errors="coerce")
            if vals.notna().any():
                rows.append({
                    "Metric": col,
                    "Mean": round(float(vals.mean()), 4),
                    "Count": int(vals.notna().sum())
                })

    return pd.DataFrame(rows)


def ordered_columns(df):
    preferred = [
        "idx", "question", "answer", "gold_answer",
        "gold_ids", "retrieved_ids", "llm_context_ids",
        "Recall@1", "Recall@3", "Recall@5",
        "Recall1", "Recall3", "Recall5",
        "MRR", "Token_F1", "token_f1",
        "EM", "ROUGE1", "ROUGE_F", "BLEU",
        "Faithfulness", "Hallucination_Risk", "Citation_Accuracy",
        "Final_Rubric_Score", "Final_Rubric_Proxy",
        "error"
    ]

    cols = [c for c in preferred if c in df.columns]
    cols += [c for c in df.columns if c not in cols]
    return cols


def run_benchmark(mode, upload_widget, use_demo_checkbox, limit_box, output_area):
    with output_area:
        clear_output()

        try:
            filename, data = read_uploaded_json(upload_widget)

            if data is None:
                if use_demo_checkbox.value:
                    filename = "built_in_demo_benchmark.json"
                    data = DEMO_BENCHMARK
                else:
                    display(HTML("""
                    <div style="padding:12px;border-radius:10px;background:#fff3cd;border:1px solid #ffe69c;color:#664d03;">
                      <b>No benchmark file selected.</b><br>
                      Upload a JSON file or enable <b>Use built-in demo benchmark</b>.
                    </div>
                    """))
                    return

            if limit_box.value and limit_box.value > 0:
                data = data[:limit_box.value]

            out_dir = BASE_DIR / "evaluation_results" / "instructor_benchmark_ui"
            out_dir.mkdir(parents=True, exist_ok=True)

            input_path = out_dir / "uploaded_instructor_benchmark.json"
            output_path = out_dir / (
                "instructor_benchmark_retrieval_only.csv"
                if mode == "retrieval"
                else "instructor_benchmark_full_rag.csv"
            )

            save_json(data, input_path)

            mode_title = "Retrieval-only Benchmark" if mode == "retrieval" else "Full RAG Benchmark"

            display(HTML(f"""
            <div style="padding:14px;border-radius:12px;background:#eef6ff;border:1px solid #b9dcff;margin-bottom:12px;">
              <h3 style="margin:0;color:#0b3a67;">Running: {mode_title}</h3>
              <p style="margin:6px 0 0 0;">
                <b>Input:</b> {filename}<br>
                <b>Questions:</b> {len(data)}<br>
                <b>Output:</b> {output_path}
              </p>
            </div>
            """))

            result = run_script(input_path, output_path, mode)

            if result.stdout.strip():
                display(HTML("<b>Script output:</b>"))
                print(result.stdout)

            if result.stderr.strip():
                display(HTML("<b>Warnings / stderr:</b>"))
                print(result.stderr)

            if result.returncode != 0:
                display(HTML(f"""
                <div style="padding:12px;border-radius:10px;background:#f8d7da;border:1px solid #f5c2c7;color:#842029;">
                  <b>Benchmark script failed.</b> Return code: {result.returncode}
                </div>
                """))
                return

            if not output_path.exists():
                display(HTML("""
                <div style="padding:12px;border-radius:10px;background:#f8d7da;border:1px solid #f5c2c7;color:#842029;">
                  <b>Benchmark finished but output CSV was not created.</b>
                </div>
                """))
                return

            df = pd.read_csv(output_path)
            summary = summarize_metrics(df)

            display(HTML("""
            <div style="padding:12px;border-radius:10px;background:#d1e7dd;border:1px solid #badbcc;color:#0f5132;margin:12px 0;">
              <b>Benchmark completed successfully.</b>
            </div>
            """))

            display(HTML("<h3 style='color:#123366;'>Summary Metrics</h3>"))

            if len(summary) > 0:
                display(summary)
            else:
                display(HTML("<p>No numeric metric columns were found. Showing full detailed results below.</p>"))

            display(HTML("<h3 style='color:#123366;'>Complete Detailed Results</h3>"))
            display(df[ordered_columns(df)])

            display(HTML("<h3 style='color:#123366;'>Saved Files</h3>"))
            print("Input JSON:", input_path)
            print("Output CSV:", output_path)

        except Exception as e:
            clear_output()
            display(HTML(f"""
            <div style="padding:12px;border-radius:10px;background:#f8d7da;border:1px solid #f5c2c7;color:#842029;">
              <b>Error:</b> {str(e)}
            </div>
            """))


def launch_benchmark_ui():
    title = HTML("""
    <div style="padding:18px;border-radius:16px;background:#f4f7fb;border:1px solid #d8e1ef;margin-bottom:16px;">
      <h2 style="margin:0;color:#102a43;">Turkish Legal RAG – Instructor Benchmark UI</h2>
      <p style="margin:8px 0 0 0;color:#334e68;font-size:14px;">
        Upload a benchmark JSON file, select a benchmark mode, and view complete evaluation results.
      </p>
    </div>
    """)

    format_box = HTML("""
    <div style="padding:14px;border-radius:12px;background:#ffffff;border:1px solid #e1e7ef;margin-bottom:14px;">
      <h4 style="margin:0 0 8px 0;color:#123366;">Expected JSON Format</h4>
      <pre style="background:#f7f9fc;border:1px solid #e1e7ef;padding:10px;border-radius:8px;white-space:pre-wrap;">[
  {
    "question": "Türk Borçlar Kanunu m.314 kapsamında ifa zamanı nasıl düzenlenmiştir?",
    "gold_answer": "Kiracı kira bedelini her ayın sonunda ödemekle yükümlüdür.",
    "gold_ids": ["turkish_law_eski_6098_turk_borclar_kanunu_m314"]
  }
]</pre>
    </div>
    """)

    mode_cards = HTML("""
    <div style="display:flex;gap:14px;margin-bottom:14px;">
      <div style="flex:1;padding:14px;border-radius:12px;background:#eef6ff;border:1px solid #b9dcff;">
        <h4 style="margin:0;color:#0b3a67;">Mode 1 — Retrieval-only Benchmark</h4>
        <p style="font-size:13px;margin:8px 0 0 0;">
          Computes retrieval metrics such as <b>Recall@1, Recall@3, Recall@5, and MRR</b>.
          It does <b>not</b> load the LLM generator, so it is faster and safer for demo.
        </p>
      </div>
      <div style="flex:1;padding:14px;border-radius:12px;background:#fff7e6;border:1px solid #ffd591;">
        <h4 style="margin:0;color:#7a4b00;">Mode 2 — Full RAG Benchmark</h4>
        <p style="font-size:13px;margin:8px 0 0 0;">
          Runs retrieval and answer generation. It can show answer-level metrics such as
          <b>Token F1</b> when gold answers are provided. Requires more GPU memory.
        </p>
      </div>
    </div>
    """)

    upload = widgets.FileUpload(
        accept=".json",
        multiple=False,
        description="Upload JSON",
        layout=widgets.Layout(width="220px")
    )

    use_demo = widgets.Checkbox(
        value=False,
        description="Use built-in demo benchmark",
        indent=False,
        layout=widgets.Layout(width="260px")
    )

    limit_box = widgets.IntText(
        value=1,
        description="Question limit:",
        style={"description_width": "initial"},
        layout=widgets.Layout(width="220px")
    )

    run_retrieval = widgets.Button(
        description="Run Retrieval-only Benchmark",
        button_style="success",
        icon="search",
        layout=widgets.Layout(width="260px", height="42px")
    )

    run_full = widgets.Button(
        description="Run Full RAG Benchmark",
        button_style="warning",
        icon="play",
        layout=widgets.Layout(width="240px", height="42px")
    )

    output = widgets.Output()

    controls = widgets.VBox([
        widgets.HBox(
            [upload, use_demo, limit_box],
            layout=widgets.Layout(gap="16px", align_items="center")
        ),
        widgets.HBox(
            [run_retrieval, run_full],
            layout=widgets.Layout(gap="16px", margin="12px 0 12px 0")
        ),
    ])

    run_retrieval.on_click(lambda _: run_benchmark("retrieval", upload, use_demo, limit_box, output))
    run_full.on_click(lambda _: run_benchmark("full", upload, use_demo, limit_box, output))

    display(title)
    display(format_box)
    display(mode_cards)
    display(controls)
    display(output)


launch_benchmark_ui()
'''

Path("benchmark_ui.py").write_text(code, encoding="utf-8")
print("benchmark_ui.py created")

benchmark_ui.py created


In [8]:
!python -m py_compile benchmark_ui.py
%run benchmark_ui.py

Output()